# 03 — Hybrid Search with HelixIndex

`HelixIndex` combines keyword search (`RelevanceIndex`) and vector search (`SimlarEngine`) and merges their rankings with Reciprocal Rank Fusion.

In this notebook we index 500 real Quora questions and compare text-only, vector-only, and hybrid results side by side.

In [ ]:
%pip install -q datasets sentence-transformers simlar

## Load dataset and compute embeddings

We use the corpus split of [`BeIR/quora`](https://huggingface.co/datasets/BeIR/quora) — real questions from Quora. We embed them with `all-MiniLM-L6-v2` (384-dim, ~80 MB download once).

In [1]:
import tempfile
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from simlar import HelixIndex, RelevanceIndex, SimlarEngine, ReciprocalRankFusion, load_from_directory

ds = load_dataset("BeIR/quora", "corpus", split="corpus[:500]")
corpus = ds["text"]
ids    = ds["_id"]

model   = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(corpus, normalize_embeddings=True, show_progress_bar=True)
vectors = vectors.astype(np.float32)  # (500, 384)

id_to_text = dict(zip(ids, corpus))

print(f"Loaded {len(corpus)} questions  |  embedding dim: {vectors.shape[1]}")
print(f"Sample: {corpus[0][:100]}")

/home/aramirez/anaconda3/envs/testing_simlar/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/aramirez/anaconda3/envs/testing_simlar/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Batches: 100%|██████████| 16/16 [00:00<00:00, 42.30it/s]

Loaded 500 questions  |  embedding dim: 384
Sample: What is the step by step guide to invest in share market in india?


## Build the index

`add()` dispatches automatically: `texts` go to `RelevanceIndex`, `vectors` go to `SimlarEngine`.

In [2]:
index = HelixIndex(
    top_k=5,
    text_k=10,
    vector_k=10,
)
index.add(ids=list(ids), texts=list(corpus), vectors=vectors)

print(f"Index size: {index.size}")

Index size: 500


## Search

Pass `query_text`, `query_vector`, or both — `HelixIndex` routes automatically and fuses whatever signals are provided.

In [3]:
# Testing: "What is the best programming language to learn first?"
QUERY_TEXT = "What is the best programming language to learn first?"
query_vec  = model.encode([QUERY_TEXT], normalize_embeddings=True).astype(np.float32)  # (1, 384)

print(f"Query: '{QUERY_TEXT}'\n")

# — Hybrid (text + vector) —
print("Hybrid:")
for r in index.search(query_text=QUERY_TEXT, query_vector=query_vec, k=5):
    print(f"  rank={r.rank}  score={r.score:.4f}  |  {id_to_text[r.id][:80]}")

# — Text only —
print("\nText only:")
for r in index.search(query_text=QUERY_TEXT, k=5):
    print(f"  rank={r.rank}  score={r.score:.4f}  |  {id_to_text[r.id][:80]}")

# — Vector only —
print("\nVector only:")
for r in index.search(query_vector=query_vec, k=5):
    print(f"  rank={r.rank}  score={r.score:.4f}  |  {id_to_text[r.id][:80]}")

Query: 'What is the best programming language to learn first?'

Hybrid:


  rank=0  score=0.5833  |  How do I learn a computer language like java?
  rank=1  score=0.5833  |  What is Java programming? How To Learn Java Programming Language ?
  rank=2  score=0.3667  |  What's the best way to start learning robotics?
  rank=3  score=0.2833  |  What are the best ways to learn French?
  rank=4  score=0.2083  |  How can I learn computer security?

Text only:
  rank=0  score=0.3333  |  What is Java programming? How To Learn Java Programming Language ?
  rank=1  score=0.2500  |  How do I learn a computer language like java?
  rank=2  score=0.2000  |  What are the best ways to learn French?
  rank=3  score=0.1667  |  What's the best way to start learning robotics?
  rank=4  score=0.1429  |  What was your first sexual experience?

Vector only:
  rank=0  score=0.3333  |  How do I learn a computer language like java?
  rank=1  score=0.2500  |  What is Java programming? How To Learn Java Programming Language ?
  rank=2  score=0.2000  |  What's the best way to start learn

## Save and load

In [4]:
with tempfile.TemporaryDirectory() as tmp:
    index.save(tmp)
    index2 = load_from_directory(tmp)
    index2._fusion = ReciprocalRankFusion()  
    print(f"Loaded index size: {index2.size}")

    # Testing: "How do I lose weight quickly?"
    QUERY_TEXT = "How do I lose weight quickly?"
    query_vec  = model.encode([QUERY_TEXT], normalize_embeddings=True).astype(np.float32)

    print(f"Query: '{QUERY_TEXT}'\n")
    for r in index2.search(query_text=QUERY_TEXT, query_vector=query_vec, k=5):
        print(f"  rank={r.rank}  score={r.score:.4f}  |  {id_to_text[r.id][:80]}")

Loaded index size: 500
Query: 'How do I lose weight quickly?'

  rank=0  score=0.3768  |  At what age, how, and where did you lose your virginity?
  rank=1  score=0.3333  |  What is the best diet for a growing decathlete?
  rank=2  score=0.2500  |  How do I utilize free time to avoid depression?
  rank=3  score=0.2500  |  At what age should someone lose their virginity?
  rank=4  score=0.2000  |  What is the quickest way to increase Instagram followers?
